In [2]:
import os

DATA_DIR = os.path.join('..', '..', 'data')
print(os.listdir(DATA_DIR))

['experiment', 'test', 'train', 'valid']


In [23]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import torchvision.transforms as T

In [16]:
image_width = 256
image_height = 128
batch_size = 128
stats = (0.06806663, 0.06942986, 0.07024706), (0.03520789, 0.03602721, 0.03545172)

### Building the Model

In [30]:
class DownSample(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=4):
        super(DownSample, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, 
                      stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2)
        )
        
    def forward(self, x):
        x = self.model(x)
        return x

class UpSample(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=4):
        super(UpSample, self).__init__()
        self.model = nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=kernel_size, 
                               stride=2, padding=1, bias=False),
            nn.InstanceNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, skip_input):
        x = self.model(x)
        x = torch.cat((x, skip_input), 1)
        return x